ID : 고유 식별자

Age : 나이

Gender : 성별 (Male: 남성, Female: 여성)

BMI : 체질량지수

ALT : 알라닌 아미노전이효소, 간 기능 관련 수치

FPG : 공복 혈당

Chol : 총콜레스테롤

Tri : 중성지방

HDL : 고밀도 지단백 콜레스테롤(HDL 콜레스테롤)

LDL : 저밀도 지단백 콜레스테롤(LDL 콜레스테롤)

DBP : 이완기 혈압


In [1]:
# EDA : 통계적 근거 제시 -> 가설검증 -> p-value

# 결측치 : 대표값으로 변경(숫자:중앙값, 평균, 범주:가장많은 빈도)
# -> sklearn 함수를 사용

# 1개의 모델 -> 2개의 모델 -> 다수의 모델(반복문)

In [2]:
# 1단계 : 데이터 수집
import pandas as pd

train = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_2_train.csv')
test = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/6_2_test.csv')

In [3]:
print(train.shape)
print(test.shape)

(301, 12)
(129, 12)


In [4]:
train.info()
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 12 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   level_0  301 non-null    int64  
 1   ID       301 non-null    int64  
 2   Age      301 non-null    int64  
 3   Gender   290 non-null    str    
 4   BMI      301 non-null    float64
 5   DBP      301 non-null    int64  
 6   FPG      301 non-null    float64
 7   Chol     301 non-null    float64
 8   Tri      301 non-null    float64
 9   HDL      301 non-null    float64
 10  LDL      301 non-null    float64
 11  ALT      301 non-null    float64
dtypes: float64(7), int64(4), str(1)
memory usage: 28.3 KB
<class 'pandas.DataFrame'>
RangeIndex: 129 entries, 0 to 128
Data columns (total 12 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   level_0  129 non-null    int64  
 1   ID       129 non-null    int64  
 2   Age      129 non-null    int64  
 3   Gender   122 non-null    s

In [5]:
train.head(20)
test.head()

,level_0,ID,Age,Gender,BMI,DBP,FPG,Chol,Tri,HDL,LDL,ALT
0,0,2856,39,NaN,22.2,83,5.30,3.63,0.79,1.400000,1.740000,13.0
1,1,3116,60,NaN,25.2,79,5.90,4.28,0.65,4.860753,4.860753,19.0
2,2,1125,33,NaN,20.7,82,5.55,3.74,0.57,1.020000,2.260000,16.0
3,3,3604,53,NaN,27.8,59,4.60,4.27,1.14,1.250000,2.230000,16.0
4,4,1334,40,NaN,25.8,113,4.66,4.92,2.12,0.890000,3.200000,87.5


In [6]:
from numpy import number
# 통계적 탐색 방법을 사용할 수도 있다. - 피어슨 상관계수
# 연속형 수치 컬럼 리스트
number_cols = train.select_dtypes('number').columns.drop('DBP')
number_cols

Index(['level_0', 'ID', 'Age', 'BMI', 'FPG', 'Chol', 'Tri', 'HDL', 'LDL',
       'ALT'],
      dtype='str')

In [7]:
from scipy.stats import pearsonr

# 귀무가설 : 두 변수간에는 선형관계가 없다 -> 변수가 ML학습에 필요없다
# 대립가설 : 두 변수간에는 선형관계가 있다
# 유의검증 수치 : 0.05
for col in number_cols:
  # r=0이면 없다 -> 귀무가설 채택
  # r!=0이면 있다 -> p>=0.05 -> 귀무가설 채택
  r, p = pearsonr(train[col], train['DBP'])
  print(f'{col}컬럼의 r은 {r:.5f}이고 p는 {p:.5f} 이다')

level_0컬럼의 r은 -0.05746이고 p는 0.32040 이다
ID컬럼의 r은 0.19453이고 p는 0.00069 이다
Age컬럼의 r은 0.22032이고 p는 0.00012 이다
BMI컬럼의 r은 0.29585이고 p는 0.00000 이다
FPG컬럼의 r은 0.11495이고 p는 0.04631 이다
Chol컬럼의 r은 0.14838이고 p는 0.00994 이다
Tri컬럼의 r은 0.11089이고 p는 0.05463 이다
HDL컬럼의 r은 0.06973이고 p는 0.22772 이다
LDL컬럼의 r은 0.16158이고 p는 0.00495 이다
ALT컬럼의 r은 0.13688이고 p는 0.01750 이다


In [8]:
# level_0, ID 삭제 : 식별자이고, 인덱스형이라
# HDL : 피이슨 검증에서 대상이지만,....... 남겨 두기로....

In [9]:
# 2. 불필요한 컬럼 삭제

X = train.drop(columns=['level_0', 'ID', 'DBP'])
y = train['DBP']
display(X.head(1))
display(y.head(1))


# 3. 데이터셋 분할 train / val / test
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=1)

X_test = test.drop(columns=['level_0', 'ID', 'DBP'])
y_test = test['DBP']
test_id = test['ID']

display(X_train.head(1))
display(y_train.head(1))

display(X_val.head(1))
display(y_val.head(1))

display(X_test.head(1))
display(y_test.head(1))
display(test_id.head(1))

,Age,Gender,BMI,FPG,Chol,Tri,HDL,LDL,ALT
0,56,NaN,22.4,5.51,4.39,1.14,1.29,3.12,15.3


0    93
Name: DBP, dtype: int64

,Age,Gender,BMI,FPG,Chol,Tri,HDL,LDL,ALT
263,43,F,25.0,5.0,5.1,3.34,1.02,2.58,27.0


263    66
Name: DBP, dtype: int64

,Age,Gender,BMI,FPG,Chol,Tri,HDL,LDL,ALT
285,61,M,19.9,4.0,5.0,0.0,4.860753,4.860753,14.0


285    81
Name: DBP, dtype: int64

,Age,Gender,BMI,FPG,Chol,Tri,HDL,LDL,ALT
0,39,NaN,22.2,5.3,3.63,0.79,1.4,1.74,13.0


0    83
Name: DBP, dtype: int64

0    2856
Name: ID, dtype: int64

In [10]:
# 4. 전처리
# Gender컬럼을 비지도학습 후 결측치를 처리 -> M/F
# M/F -> 01/10/00 -> OneHot인코딩

# 데이터를 먼저 분할한 뒤, 전처리 기준을 train만으로 학습 후 val, test에 적용
# 인코딩, 스케일링, 결측치, ...시 전체 데이터로 학습하면
# 검증·테스트 데이터의 정보가 학습 과정에 섞이는 데이터 누수가 생길 수 있다.

# fit_transform() 메서드 사용시 주의 : train데이터셋에만 하는게 좋다. -> 전처리는 데이터 분할 후에 하자

# display(X_train['Gender'].value_counts(dropna=False))

from sklearn.impute import SimpleImputer

# 1) imputer
imputer = SimpleImputer(strategy='most_frequent').set_output(transform="pandas")
X_train_imp = imputer.fit_transform(X_train[['Gender']]) # cat_cols = X_train.select_dtypes('object).columns
X_val_imp = imputer.transform(X_val[['Gender']])
X_test_imp = imputer.transform(X_test[['Gender']])
# X_train_cat.value_counts(dropna=False)

# 2) encoder
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore').set_output(transform='pandas')
X_train_en = encoder.fit_transform(X_train_imp)
X_val_en = encoder.transform(X_val_imp)
X_test_en = encoder.transform(X_test_imp)

X_train_en.shape


(210, 2)

In [11]:
# X_traina에서 숫자형 + X_train_en
# X_traina에서 범주형 drop후 + X_train_en
num_cols2 = X_train.select_dtypes('number').columns
print(num_cols2)

X_train_pre = pd.concat([X_train[num_cols2], X_train_en], axis=1)
X_val_pre = pd.concat([X_val[num_cols2], X_val_en], axis=1)
X_test_pre = pd.concat([X_test[num_cols2], X_test_en], axis=1)

display(X_train_pre.head())
display(X_val_pre.head())
display(X_test_pre.head())

Index(['Age', 'BMI', 'FPG', 'Chol', 'Tri', 'HDL', 'LDL', 'ALT'], dtype='str')


,Age,BMI,FPG,Chol,Tri,HDL,LDL,ALT,Gender_F,Gender_M
263,43,25.0,5.00,5.10,3.34,1.020000,2.580000,27.0,1.0,0.0
19,77,23.6,5.14,6.07,0.99,1.220000,3.710000,20.2,1.0,0.0
14,42,25.1,4.74,4.30,0.80,1.540000,2.690000,31.0,1.0,0.0
91,42,23.4,5.20,4.02,1.86,1.640000,1.810000,52.0,1.0,0.0
211,77,31.0,6.40,5.14,2.07,4.860753,4.860753,26.0,1.0,0.0


,Age,BMI,FPG,Chol,Tri,HDL,LDL,ALT,Gender_F,Gender_M
285,61,19.9,4.00,5.00,0.00,4.860753,4.860753,14.0,0.0,1.0
248,73,21.3,4.98,4.46,1.85,1.250000,2.610000,19.2,0.0,1.0
150,66,29.1,4.96,6.41,2.67,1.710000,3.360000,46.3,0.0,1.0
217,37,30.5,5.07,7.88,2.59,1.710000,4.270000,59.3,1.0,0.0
107,43,23.4,5.00,4.10,0.62,1.300000,2.370000,48.0,1.0,0.0


,Age,BMI,FPG,Chol,Tri,HDL,LDL,ALT,Gender_F,Gender_M
0,39,22.2,5.30,3.63,0.79,1.400000,1.740000,13.0,1.0,0.0
1,60,25.2,5.90,4.28,0.65,4.860753,4.860753,19.0,1.0,0.0
2,33,20.7,5.55,3.74,0.57,1.020000,2.260000,16.0,1.0,0.0
3,53,27.8,4.60,4.27,1.14,1.250000,2.230000,16.0,1.0,0.0
4,40,25.8,4.66,4.92,2.12,0.890000,3.200000,87.5,1.0,0.0


In [12]:
# from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score

model = RandomForestRegressor(random_state=1)

model.fit(X_train_pre, y_train)

pred_val = model.predict(X_val_pre)

rmse = root_mean_squared_error(y_val, pred_val)
r2 = r2_score(y_val, pred_val)
print(rmse)
print(r2)

12.521506027104452
0.08443654596072792


In [13]:
pred_test = model.predict(X_test_pre)

print(pred_test[:10])
print(y_test[:10])

r2_test = r2_score(y_test, pred_test)
print(r2_test)

[73.72 80.13 77.14 79.5  75.9  81.13 80.37 78.2  77.35 75.11]
0     83
1     79
2     82
3     59
4    113
5     68
6     93
7     68
8     74
9     71
Name: DBP, dtype: int64
-0.024416841581248017
